In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parents[0]  # notebooks/ -> repo root
sys.path.insert(0, str(REPO_ROOT / "src"))

import recipe_wrangler


In [2]:
from pathlib import Path
import recipe_wrangler

REPO_ROOT = Path(recipe_wrangler.__file__).resolve().parents[2]  # package -> src -> repo
REPO_ROOT

PosixPath('/home/karvanitis/RecipeWrangler-Backend')

## Check Chromadb Databases

In [ ]:
from recipe_wrangler.utils.chroma_client import get_chroma_client, CHROMA_HOST, CHROMA_PORT

client = get_chroma_client()  # defaults: host=localhost, port=8000
collections = client.list_collections()

print(f"Collections at {CHROMA_HOST}:{CHROMA_PORT}")
for col in collections:
    print(f"- {col.name} (count={col.count()})")


## Check Utilities

In [ ]:
from recipe_wrangler.utils.query_chromadb import get_ingredient_embedding

get_ingredient_embedding("low-fat gouda")

In [ ]:
from recipe_wrangler.utils.query_chromadb import query_sustainability_db

result = query_sustainability_db("low-fat gouda")
print(result)

In [ ]:
from recipe_wrangler.utils.query_chromadb import query_nutritional_db_irish

result = query_nutritional_db_irish("tomatoe")

print(result)

## Parse Recipe Tool

In [ ]:
from recipe_wrangler.tools.parse_recipe_tool import parse_recipe_tool
import json

raw_text = """
Garlic Butter Shrimp

Ingredients (2 servings):

200g shrimp (peeled & deveined)

2 tbsp butter

3 garlic cloves, minced

Juice of half a lemon

Salt & pepper to taste

Fresh parsley (optional)

Instructions:

Heat butter in a skillet over medium heat.

Add garlic, cook 30 seconds until fragrant.

Toss in shrimp, season with salt & pepper.

Cook 2–3 minutes per side until pink.

Squeeze lemon juice on top and sprinkle parsley.

Serve with rice, pasta, or crusty bread.
"""

recipe_dict = parse_recipe_tool.invoke({"recipe": raw_text})

print(json.dumps(recipe_dict, indent=2))

## Check Weight Ingredient Tool

In [ ]:
# Weight Calculator Tool

from recipe_wrangler.tools.ingredient_weight_tool import ingredient_weight_tool_usda

ingredient_names = recipe_dict['ingredient_names']
measurements = recipe_dict['measurements']

weights = ingredient_weight_tool_usda.invoke({
    "ingredient_names": ingredient_names,
    "measurements": measurements,
    "return_details": True
})
print(weights)


## Check Embeddings Tool

In [ ]:
# Embeddings tool

from recipe_wrangler.tools.ingredient_embeddings_tool import ensure_ingredients_in_collection
payload = ensure_ingredients_in_collection.invoke({
    "ingredient_names": ["garlic", "olive oil", "pasta", "low-fat gouda", "canned cherries"],
    "state": {"persist_path": "chroma_db", "collection_name": "ingredients", "debug": True}
})

print(payload)

In [ ]:
# Check the new added ingredient in the ingredient collection

from recipe_wrangler.utils.query_chromadb import query_sustainability_db, query_nutritional_db_irish, query_ingredients_db

query_ingredients_db("canned cherries")

## Check Sustainability Tool

In [ ]:
from recipe_wrangler.tools.sustainability_calculator import sustainability_tool_chroma

In [ ]:
# --- Run a real test call via the tool interface (.invoke) ---

result = sustainability_tool_chroma.invoke({
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",      # typo on purpose (should match tomato)
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": [
        "400.0g",      # typo on purpose (should match tomato)
        "30.0g",
        "80.0g",
        "10.0g",
        "320.0g",
        "40.0g",
        "5.0g",
    ],
    "weights": [
        400.0,      # typo on purpose (should match tomato)
        30.0,
        80.0,
        10.0,
        320.0,
        40.0,
        5.0,
    ],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,      # your requested similarity threshold
})


In [ ]:
result

## Check Nutritional Tool

In [ ]:
from recipe_wrangler.tools.nutritional_calculator import nutritional_tool_chroma

In [ ]:
# --- Run a real test call via the tool interface (.invoke) ---

result = nutritional_tool_chroma.invoke({
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",      # typo on purpose (should match tomato)
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": [
        "400.0g",      # typo on purpose (should match tomato)
        "30.0g",
        "80.0g",
        "10.0g",
        "320.0g",
        "40.0g",
        "5.0g",
    ],
    "weights": [
        400.0,      # typo on purpose (should match tomato)
        30.0,
        80.0,
        10.0,
        320.0,
        40.0,
        5.0,
    ],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,      # your requested similarity threshold
})


In [ ]:
result

## Check Profiling Tool

In [ ]:
from recipe_wrangler.tools.recipe_profiling_tool import Recipe_Profiling_Tool

In [ ]:
payload = {
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": ["400.0g", "30.0g", "80.0g", "10.0g", "320.0g", "40.0g", "5.0g"],
    "weights": [400.0, 30.0, 80.0, 10.0, 320.0, 40.0, 5.0],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,
}

profile = Recipe_Profiling_Tool(payload)
from pprint import pprint
pprint(profile)

## Check Profiling Chain

In [ ]:
from recipe_wrangler.tools.recipe_profiling_chain import Recipe_Profiling_Chain

In [ ]:
sample_recipe = """
Pasta al Pomodoro
Serves 4

Ingredients:
- 400 g tomatoes (ripe)
- 30 g olive oil
- 1 small carrot, finely chopped
- 80 g onion, finely chopped
- 10 g garlic, minced
- 320 g spaghetti
- 40 g parmesan, grated
- 5 g basil leaves
- Salt to taste

Directions:
1) Sweat onion in olive oil until translucent. Add garlic briefly.
2) Add chopped tomatoes, simmer 15–20 min. Season with salt.
3) Cook spaghetti al dente; toss with sauce.
4) Serve with basil and parmesan.
"""

res = Recipe_Profiling_Chain.invoke({"recipe_text": sample_recipe, "debug": True})
print(res.keys())              
print(res.get("recipe_profile_totals"))

In [ ]:
res

## LangChain T2C tool

In [ ]:
from recipe_wrangler.tools.text2cypher import RecipeSearchApp
import os

app = RecipeSearchApp(
    neo4j_uri=os.environ["NEO4J_URI"]
)

result = app.invoke("Give me a recipe with chicken")
print(result['results'])
print(result['cypher_statement'])

In [ ]:
result

## Fetch Recipe Info

In [1]:
from recipe_wrangler.tools.fetch_recipe_info import fetch_recipe_info

info = fetch_recipe_info("00010c7867")

print(info)

{'recipe_id': '00010c7867', 'title': "Grandmommy's Mexicali Meatloaf", 'ingredients': [{'name': 'Mexican - style corn', 'measurement': '1 can'}, {'name': 'butter', 'measurement': '3 tablespoons'}, {'name': 'cheddar cheese', 'measurement': '1 cup'}, {'name': 'chili powder', 'measurement': '1 teaspoon'}, {'name': 'egg', 'measurement': '1'}, {'name': 'flour', 'measurement': '3 tablespoons'}, {'name': 'green peppers', 'measurement': '2'}, {'name': 'ground beef', 'measurement': '1 1/2'}, {'name': 'milk', 'measurement': '2 cups'}, {'name': 'oats', 'measurement': '3/4'}, {'name': 'onions', 'measurement': '2 teaspoons'}, {'name': 'pepper', 'measurement': '1/4'}, {'name': 'salt', 'measurement': '2 teaspoons'}, {'name': 'tomato juice', 'measurement': '1/2'}], 'instructions': ['Combine beef, oats, tomato juice, egg, 1 tsp salt, pepper, chili powder, and onion in a large bowl.', 'Do not overmix.', 'Pack into a 9 inch square baking dish.', 'Bake at 350 for 20 minutes, drain juices.', 'Melt butter o